In [1]:
from google.colab import userdata
API_KEY = userdata.get('YOUTUBE_API_KEY')

In [2]:
!pip -q install google-api-python-client

import re, time, random
import numpy as np, pandas as pd
from datetime import datetime, timedelta, timezone
from difflib import SequenceMatcher
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.colab import userdata, files

youtube = build("youtube", "v3", developerKey=userdata.get("YOUTUBE_API_KEY"))

def run(call):
    """Run an API call. On failure print only the status (never the URL, so the key can't leak)."""
    try:
        return call.execute()
    except HttpError as e:
        print("   API error", e.resp.status, getattr(e, "reason", ""))
        return None

t = run(youtube.channels().list(part="snippet", forHandle="@YouTube"))
print("Key OK ->", t["items"][0]["snippet"]["title"] if t else "FAILED")

Key OK -> YouTube


In [9]:
!pip -q install google-api-python-client

import numpy as np, pandas as pd
from datetime import datetime, timedelta, timezone
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.colab import userdata, files

TARGET_VIDEOS     = 150
START_PER_CHANNEL = 5       # first try 5 per channel
MAX_PER_CHANNEL   = 10      # raise up to this if the total is below the target
WINDOW_MONTHS     = 24      # look back about 2 years
MIN_AGE_DAYS      = 30      # skip videos younger than 30 days (views not settled)
MIN_SUBS          = 50_000
ALLOW_SHORTS      = False   # False = skip videos of 3 minutes or less
COUNT_FACES       = True    # adds a face_count column (Haar cascade)
MAX_FACES         = None    # e.g. 3 keeps only thumbnails with 3 faces or fewer

youtube = build("youtube", "v3", developerKey=userdata.get("YOUTUBE_API_KEY"))

def run(call):
    try:
        return call.execute()
    except HttpError as e:                       # never print the URL (it contains the key)
        print("   API error", e.resp.status, getattr(e, "reason", ""))
        return None

# ---------------- your channels ----------------
HANDLES = ["BeardMeatsFood","MattStonie","RandySantel","FuriousPete","ErikTheElectric",
 "LeahShutkever","RainaHuang","ZermattNeo","WakeNBite","FoodieWe","NickDiGiovanni",
 "JoshuaWeissman","GugaFoods","BayashiTV_","SortedFood","JOLLY","TryGuys","GoodMythicalMorning",
 "Sidemen","BetaSquad","MrBeast","DudePerfect","HowRidiculous","Airrack","StokesTwins",
 "RyanTrahan","MarkRober","MichelleKhare","YesTheory","MaxFosh","Niko","Ludwig","Jesser",
 "ZachKing","TheBucketListFamily",
 # the last five had no handle in your list, so these are guesses; unresolved ones are just skipped
 "KatinaEatsKilos","MaxvsFood","JoelHansen","FreakEating","TheFoodieWe"]

pool, seen = [], set()
for h in HANDLES:
    r = run(youtube.channels().list(part="snippet,statistics", forHandle=h))
    time.sleep(0.1)
    if not (r and r.get("items")):
        print("not found:", h); continue
    it = r["items"][0]; st = it["statistics"]
    subs = int(st.get("subscriberCount", 0))
    if it["id"] in seen: continue
    seen.add(it["id"])
    if subs < MIN_SUBS:
        print("too small:", it["snippet"]["title"], subs); continue
    pool.append({"channel_id": it["id"], "channel_name": it["snippet"]["title"],
                 "subscribers": subs, "channel_video_count": int(st.get("videoCount", 0))})
print(len(pool), "channels ready")

# ---------------- helpers ----------------
now = datetime.now(timezone.utc)
end = now - timedelta(days=MIN_AGE_DAYS)
since = end - timedelta(days=30.4 * WINDOW_MONTHS)

def iso_to_sec(d):
    m = re.fullmatch(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", d or "")
    if not m: return 0
    h, mi, s = (int(x) if x else 0 for x in m.groups()); return h*3600 + mi*60 + s

def get_uploads(cid, max_pages=30):
    pl, out, tok = "UU" + cid[2:], [], None
    for _ in range(max_pages):
        r = run(youtube.playlistItems().list(part="contentDetails,snippet", playlistId=pl,
                                             maxResults=50, pageToken=tok))
        if not r: break
        stop = False
        for it in r["items"]:
            if it["snippet"].get("title") in ("Private video", "Deleted video"): continue
            ts = it["contentDetails"].get("videoPublishedAt") or it["snippet"].get("publishedAt")
            if not ts: continue
            pub = datetime.fromisoformat(ts.replace("Z", "+00:00"))
            if pub > end: continue
            if pub < since: stop = True; break
            out.append((it["contentDetails"]["videoId"], pub))
        tok = r.get("nextPageToken")
        if stop or not tok: break
    return out

def drop_shorts(ups):
    if ALLOW_SHORTS or not ups: return ups
    keep = []
    for i in range(0, len(ups), 50):
        chunk = ups[i:i+50]
        r = run(youtube.videos().list(part="contentDetails", id=",".join(v for v, _ in chunk)))
        dur = {x["id"]: iso_to_sec(x["contentDetails"].get("duration")) for x in (r or {}).get("items", [])}
        keep += [(v, p) for v, p in chunk if dur.get(v, 0) > 180]
    return keep

def pick_even(ups, k):
    """k videos spread evenly over time."""
    ups = sorted(ups, key=lambda x: x[1])
    if len(ups) <= k: return [v for v, _ in ups]
    idx = np.linspace(0, len(ups) - 1, k).round().astype(int)
    return [ups[i][0] for i in idx]

# ---------------- read uploads once ----------------
avail = {}
for ch in pool:
    avail[ch["channel_id"]] = drop_shorts(get_uploads(ch["channel_id"]))
    print(f"{ch['channel_name']}: {len(avail[ch['channel_id']])} usable videos in window")
    time.sleep(0.1)

# ---------------- raise videos-per-channel until the target is reached ----------------
for k in range(START_PER_CHANNEL, MAX_PER_CHANNEL + 1):
    selected = {cid: pick_even(u, k) for cid, u in avail.items()}
    total = sum(len(v) for v in selected.values())
    if total >= TARGET_VIDEOS: break
print(f"\nUsing up to {k} videos per channel -> {total} videos selected")

# ---------------- fetch details for the selected videos ----------------
info = {c["channel_id"]: c for c in pool}
ids = [(cid, v) for cid, vs in selected.items() for v in vs]
rows = []
for i in range(0, len(ids), 50):
    chunk = ids[i:i+50]
    r = run(youtube.videos().list(part="snippet,statistics,contentDetails",
                                  id=",".join(v for _, v in chunk)))
    for v in (r or {}).get("items", []):
        c = info[v["snippet"]["channelId"]]
        th = v["snippet"]["thumbnails"]
        thumb = (th.get("maxres") or th.get("high") or th.get("medium") or th.get("default"))["url"]
        st = v["statistics"]
        pub = datetime.fromisoformat(v["snippet"]["publishedAt"].replace("Z", "+00:00"))
        rows.append({
            "channel_name":        c["channel_name"],
            "channel_id":          c["channel_id"],
            "video_id":            v["id"],
            "title":               v["snippet"]["title"],
            "thumbnail":           thumb,
            "published_at":        v["snippet"]["publishedAt"],
            "views":               int(st["viewCount"]) if "viewCount" in st else None,
            "likes":               int(st["likeCount"]) if "likeCount" in st else None,
            "comments":            int(st["commentCount"]) if "commentCount" in st else None,
            "duration_sec":        iso_to_sec(v["contentDetails"].get("duration")),
            "age_days":            (now - pub).days,
            "tags_count":          len(v["snippet"].get("tags", [])),
            "category_id":         v["snippet"].get("categoryId"),
            "subscribers":         c["subscribers"],
            "channel_video_count": c["channel_video_count"],
        })
    time.sleep(0.2)

final = pd.DataFrame(rows).drop_duplicates("video_id")


final = final.sort_values(["channel_name", "published_at"]).reset_index(drop=True)
final.to_csv("challenge_videos_final.csv", index=False, encoding="utf-8-sig")

print(f"\nChannels: {final['channel_id'].nunique()}   Videos: {len(final)}")
print("Missing views:", final["views"].isna().sum())
print(final["channel_name"].value_counts())
pd.set_option("display.max_rows", None); pd.set_option("display.max_colwidth", 50)
display(final)
files.download("challenge_videos_final.csv")

not found: RyanTrahan
not found: TheFoodieWe
38 channels ready
BeardMeatsFood: 108 usable videos in window
Matt Stonie: 14 usable videos in window
Randy Santel: 137 usable videos in window
Furious Pete: 1 usable videos in window
ErikTheElectric: 4 usable videos in window
Leah Shutkever: 85 usable videos in window
Raina Huang: 273 usable videos in window
Zermatt Neo: 201 usable videos in window
Wake'N'Bite: 313 usable videos in window
Foodie We: 98 usable videos in window
Nick DiGiovanni: 32 usable videos in window
Joshua Weissman: 56 usable videos in window
Guga Foods: 121 usable videos in window
Bayashi TV: 50 usable videos in window
Sorted Food: 355 usable videos in window
JOLLY: 114 usable videos in window
The Try Guys: 144 usable videos in window
Good Mythical Morning: 519 usable videos in window
Sidemen: 107 usable videos in window
Beta Squad: 42 usable videos in window
MrBeast: 47 usable videos in window
Dude Perfect: 40 usable videos in window
How Ridiculous: 50 usable videos in

,channel_name,channel_id,video_id,title,thumbnail,published_at,views,likes,comments,duration_sec,age_days,tags_count,category_id,subscribers,channel_video_count
0,Airrack,UCyps-v4WNjWDnYRKmZ4BUGw,FTMHc-swSMQ,GTA In Real Life Prank,https://i.ytimg.com/vi/FTMHc-swSMQ/maxresdefau...,2024-11-12T20:33:55Z,4421476,113304,2286,901,728,0,24,19600000,271
1,Airrack,UCyps-v4WNjWDnYRKmZ4BUGw,sY06MKwC1P4,I Secretly Hid in Celebrity TikToks,https://i.ytimg.com/vi/sY06MKwC1P4/maxresdefau...,2025-06-15T15:42:34Z,10832956,174757,9801,1735,514,0,24,19600000,271
2,Airrack,UCyps-v4WNjWDnYRKmZ4BUGw,CPOaelxGivM,I Secretly Lived In MrBeast's House,https://i.ytimg.com/vi/CPOaelxGivM/maxresdefau...,2025-11-08T17:35:56Z,23481914,595998,9556,1807,368,0,24,19600000,271
3,Airrack,UCyps-v4WNjWDnYRKmZ4BUGw,PpDdsEBrGog,I Cheated Until the Casino Banned Me,https://i.ytimg.com/vi/PpDdsEBrGog/maxresdefau...,2026-04-29T15:32:13Z,17665520,276436,6173,1694,196,0,24,19600000,271
4,Airrack,UCyps-v4WNjWDnYRKmZ4BUGw,_LHh3BS9Xx4,I Hid In MrBeast's Grocery Store,https://i.ytimg.com/vi/_LHh3BS9Xx4/maxresdefau...,2026-10-04T15:44:46Z,25007162,264151,5553,1440,38,0,24,19600000,271
5,Bayashi TV,UCbCJmNKAL85O1VFeD6Wj60g,I9M-kp4CDe8,Best of Bayashi Foods - Onigiri !! 川場村の美味しい食材を...,https://i.ytimg.com/vi/I9M-kp4CDe8/maxresdefau...,2024-12-13T11:00:21Z,93969,2395,134,547,698,9,24,37500000,1750
6,Bayashi TV,UCbCJmNKAL85O1VFeD6Wj60g,A3L0wiHvrJw,Best of Bayashi Foods | MUKBANG | COOKING | ASMR,https://i.ytimg.com/vi/A3L0wiHvrJw/hqdefault.jpg,2025-03-03T11:00:16Z,342398,3051,144,2768,618,9,24,37500000,1750
7,Bayashi TV,UCbCJmNKAL85O1VFeD6Wj60g,vHUbmzjzOaw,Best of Bayashi Foods | MUKBANG | COOKING | ASMR,https://i.ytimg.com/vi/vHUbmzjzOaw/maxresdefau...,2025-04-05T11:01:13Z,762240,4685,168,3317,585,9,24,37500000,1750
8,Bayashi TV,UCbCJmNKAL85O1VFeD6Wj60g,a0_VwsrR9hQ,Lifehack Collection,https://i.ytimg.com/vi/a0_VwsrR9hQ/maxresdefau...,2026-08-07T09:00:28Z,280072,2121,102,725,96,9,24,37500000,1750
9,Bayashi TV,UCbCJmNKAL85O1VFeD6Wj60g,fcdgu6bkq6c,Best of Bayashi Foods | MUKBANG | COOKING | ASMR,https://i.ytimg.com/vi/fcdgu6bkq6c/maxresdefau...,2026-10-11T11:32:44Z,237319,2315,83,3833,31,8,24,37500000,1750


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>